# 🔬 Cervexa - VIA Cervical Cancer Detection Model Training

**Panduan:**
1. Pastikan Runtime menggunakan **GPU** (Runtime > Change runtime type > GPU)
2. Jalankan setiap cell dari atas ke bawah
3. Setelah selesai, download file `via_model.tflite` dan `via_model.h5`
4. Salin file `.tflite` ke `cervexa/app/src/main/assets/` dan `cervexa/ml/`

**Dataset:** Intel & MobileODT Cervical Cancer Screening (Kaggle) + IARC VIA Atlas

**Estimasi waktu training:** 15-30 menit (dengan GPU)

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q kaggle tensorflow pillow numpy matplotlib scikit-learn

import tensorflow as tf
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## Step 2: Upload Kaggle API Key

**Cara mendapatkan `kaggle.json`:**
1. Buka https://www.kaggle.com
2. Klik foto profil > Settings
3. Scroll ke bagian **API** > klik **Create New Token**
4. File `kaggle.json` akan ter-download otomatis
5. Upload file tersebut di cell di bawah ini

In [ ]:
from google.colab import files
import os

# Upload kaggle.json API key
print('Upload file kaggle.json Anda:')
uploaded = files.upload()

# Setup kaggle credentials
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle API key berhasil dikonfigurasi!')

## Step 3: Download Dataset dari Kaggle

In [ ]:
# Download Intel & MobileODT Cervical Cancer Screening Dataset
# Dataset berisi ~6700 gambar serviks terkategorisasi
!kaggle competitions download -c intel-mobileodt-cervical-cancer-screening -f train.7z

# Install 7zip untuk extract
!apt-get install -q p7zip-full

# Extract dataset
print('Extracting dataset...')
!7z x train.7z -o/content/kaggle_data/ -y > /dev/null
print('✅ Dataset berhasil di-extract!')

# Lihat struktur folder
!find /content/kaggle_data -type d | head -20

## Step 4: Juga Download Gambar Klinis dari IARC

In [ ]:
import requests
import time
from pathlib import Path

# Gambar IARC VIA Atlas yang sudah terkurasi
IARC_NORMAL = [
    'AFC0a', 'AJL0a', 'AGY0a', 'AMF0a', 'AIC0a', 'AMZ0a', 'ALC0a', 'AKT0a',
    'AJG0a', 'AIA0a', 'ANP0a', 'AGM0a', 'AJW0a', 'AID0a', 'ANV0a', 'AFJ0a',
    'ANS0a', 'AJZ0a', 'AIO1a', 'AHR0a', 'ALN0a', 'AJC0a', 'ALO0a', 'AIR0a',
    'AGT0a', 'AHN0a', 'AKQ0a', 'AIM0a', 'AKH0a', 'ADH0a', 'ALU0a', 'AND0a',
    'AME0a', 'ALK0a', 'ALR0a', 'AFK0a', 'AFN0a', 'AMG0a', 'AMX0a', 'AHP0a',
    'AJA1a', 'AKA0a', 'ANM0a', 'AMU0a', 'ANC0a', 'ANA0a', 'AKC0a', 'ALB0a',
    'AFV0a', 'AHD0a', 'AGW0a', 'AGW1a', 'AMK0a', 'AMK1a', 'AFH1a', 'ANC1a',
    'AIH0a', 'AIH1a', 'AHT1a', 'AGV0a', 'AGV1a', 'AMT0a', 'AMT1a', 'AIL0a',
    'AIL1a'
]

IARC_ABNORMAL = [
    'ADM0a', 'AEP0a', 'AJP0a', 'ADO0a', 'ACL0a', 'ABM0a', 'ABA0a', 'AEN0a',
    'ACX0a', 'ADJ0a', 'AAO0a', 'ADQ0a', 'ABO0a', 'ABH0a', 'ADZ0a', 'ACF0a',
    'ACO0a', 'ABJ0a', 'ABK0a', 'ACV0a', 'ABB0a', 'ABC0a', 'ADB0a', 'ABY0a',
    'ACZ0a', 'ABL0a', 'ABU0a', 'ACY0a', 'AEI0a', 'ADU0a', 'AEH0a', 'ADI0a',
    'ABV0a', 'ABW0a', 'AAF0a', 'ACI0a', 'ACR0a', 'ABT0a', 'ACS0a', 'AEE0a',
    'ABZ0a', 'ABX0a', 'AEJ0a', 'AEL0a', 'ADN0a', 'ABP1a', 'AEB1a', 'ABR1a',
    'AEC1a', 'ADC0a', 'ADT0a', 'ADV0a', 'ADD0a', 'ADF0a', 'AEP1a', 'AJP1a',
    'ADO1a', 'ACS1a', 'AEV0a', 'AEV1a', 'AJN1a', 'AJW0a', 'AJW1a', 'AJZ0a',
    'AJZ1a', 'AKZ1a'
]

BASE_URL = 'https://screening.iarc.fr/viavilipic/'
IARC_DIR = Path('/content/iarc_data')
IARC_DIR.joinpath('normal').mkdir(parents=True, exist_ok=True)
IARC_DIR.joinpath('abnormal').mkdir(parents=True, exist_ok=True)

def download_iarc(ids, label):
    downloaded = 0
    for filename in ids:
        url = f'{BASE_URL}{filename}.jpg'
        filepath = IARC_DIR / label / f'{filename}.jpg'
        if filepath.exists():
            continue
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                filepath.write_bytes(r.content)
                downloaded += 1
            time.sleep(0.1)
        except:
            pass
    print(f'  Downloaded {downloaded} new {label} images from IARC')

print('Downloading IARC images...')
download_iarc(IARC_NORMAL, 'normal')
download_iarc(IARC_ABNORMAL, 'abnormal')
print('✅ IARC download done!')

## Step 5: Siapkan Dataset (Gabung & Labeling)

In [ ]:
import shutil
from pathlib import Path

DATASET_DIR = Path('/content/dataset_combined')
(DATASET_DIR / 'normal').mkdir(parents=True, exist_ok=True)
(DATASET_DIR / 'abnormal').mkdir(parents=True, exist_ok=True)

# --- 1. Tambahkan gambar IARC ---
for label in ['normal', 'abnormal']:
    for f in (IARC_DIR / label).glob('*.jpg'):
        shutil.copy2(f, DATASET_DIR / label / f.name)

# --- 2. Labeling gambar dari Kaggle ---
# Intel dataset: Type 1, 2, 3 = Normal (anatomi normal, bukan kanker)
# Untuk VIA: kita labelkan Type 1/2/3 sebagai 'normal'
kaggle_train = Path('/content/kaggle_data/train')

if kaggle_train.exists():
    # Cari semua subfolder (Type_1, Type_2, Type_3)
    type_folders = [d for d in kaggle_train.iterdir() if d.is_dir()]
    print(f'Kaggle dataset folders: {[d.name for d in type_folders]}')
    
    count = 0
    for folder in type_folders:
        for img in list(folder.glob('*.jpg'))[:500]:  # max 500 per type untuk balance
            shutil.copy2(img, DATASET_DIR / 'normal' / f'kaggle_{folder.name}_{img.name}')
            count += 1
    print(f'Copied {count} Kaggle images as Normal')
else:
    print('Kaggle dataset not found, using IARC only')

# Hitung total
n_normal = len(list((DATASET_DIR / 'normal').glob('*.jpg')))
n_abnormal = len(list((DATASET_DIR / 'abnormal').glob('*.jpg')))
print(f'\n📊 Dataset Summary:')
print(f'  Normal   : {n_normal} images')
print(f'  Abnormal : {n_abnormal} images')
print(f'  Total    : {n_normal + n_abnormal} images')

## Step 6: Training Model (EfficientNetV2B0 + Transfer Learning)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np

DATASET_DIR = '/content/dataset_combined'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Bisa lebih besar karena pakai GPU
EPOCHS_PHASE1 = 30
EPOCHS_PHASE2 = 50
MODEL_SAVE_PATH = '/content/via_model.h5'
TFLITE_SAVE_PATH = '/content/via_model.tflite'

# Load dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_ds.class_names
print(f'Classes: {class_names}')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Build Model
base_model = tf.keras.applications.EfficientNetV2B0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    # Data Augmentation
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.3),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    # Base Model
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor='val_loss', verbose=1),
    EarlyStopping(patience=8, monitor='val_loss', restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

# --- Phase 1: Train classification head ---
print('\n[Phase 1] Training classification head...')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks
)

# --- Phase 2: Fine-tuning ---
print('\n[Phase 2] Fine-tuning top layers...')
base_model.trainable = True
for layer in base_model.layers[:-50]:  # Freeze semua kecuali 50 layer terakhir
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks
)

print('\n✅ Training selesai!')

## Step 7: Evaluasi Model (Confusion Matrix & Accuracy)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Evaluasi pada data validasi
results = model.evaluate(val_ds, verbose=1)
print(f'\n📊 Hasil Evaluasi Akhir:')
print(f'  Loss      : {results[0]:.4f}')
print(f'  Accuracy  : {results[1]*100:.2f}%')
print(f'  Precision : {results[2]*100:.2f}%')
print(f'  Recall    : {results[3]*100:.2f}%')

# Confusion Matrix
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred.extend((preds.flatten() > 0.5).astype(int))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Normal', 'Predicted Abnormal'],
            yticklabels=['Actual Normal', 'Actual Abnormal'])
plt.title('Confusion Matrix - Cervexa VIA Model')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

print('\n📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

acc1 = history1.history['accuracy']
val_acc1 = history1.history['val_accuracy']
all_acc = acc1 + history2.history['accuracy']
all_val_acc = val_acc1 + history2.history['val_accuracy']

ax1.plot(all_acc, label='Training Accuracy')
ax1.plot(all_val_acc, label='Validation Accuracy')
ax1.axvline(x=len(acc1), color='r', linestyle='--', label='Fine-tuning Start')
ax1.set_title('Model Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
ax2.plot(all_loss, label='Training Loss')
ax2.plot(all_val_loss, label='Validation Loss')
ax2.axvline(x=len(history1.history['loss']), color='r', linestyle='--', label='Fine-tuning Start')
ax2.set_title('Model Loss')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150)
plt.show()

## Step 8: Konversi ke TFLite (Optimized for Mobile)

In [ ]:
# Load best model
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)

# Convert to TFLite with quantization
print('Converting to TFLite...')
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(TFLITE_SAVE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f'✅ TFLite model saved: {size_mb:.2f} MB')

## Step 9: Download File Model

In [ ]:
from google.colab import files

print('Downloading model files...')

# Download TFLite (untuk Android app)
files.download('/content/via_model.tflite')

# Download H5 (backup full model)
files.download('/content/via_model.h5')

# Download evaluation charts
files.download('/content/confusion_matrix.png')
files.download('/content/training_history.png')

print('\n✅ Semua file berhasil di-download!')
print('\n📌 Langkah selanjutnya:')
print('1. Salin via_model.tflite ke: cervexa/ml/')
print('2. Salin via_model.tflite ke: cervexa/app/src/main/assets/')
print('3. Build ulang aplikasi Android')